In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Загружаем датасет

In [ ]:
import json

data_file = "/content/drive/MyDrive/TelegramData/clean_dataset.json"


with open(data_file, 'r') as f:
  data = json.load(f)

len(data), type(data)

In [ ]:
# зависимости
!pip install transformers sentence-transformers

import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split

from transformers import AutoModelForCausalLM, Trainer, TrainingArguments
from datasets import Dataset

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available else "cpu")
device

In [ ]:
# model_embs = SentenceTransformer("cointegrated/rubert-tiny2")
# model_embs.cuda()


# all_comments = [c for com_list in comments
#                   for c in com_list]

# len(all_comments)

In [ ]:
corpus = [(post, coms) for post in data.keys() for coms in data[post]]
len(corpus)

In [ ]:
corpus[1]

In [ ]:
# кодируем в эмбеддинги комментарии посты
# comment_embs = model_embs.encode(all_comments, convert_to_numpy=True)
# post_embs = model_embs.encode(posts, convert_to_numpy=True)

# кластеризируем (получим кластеры на основе настроений комментариев.
# можно использовать отдельные модели для сентимент анализа и разбить комментарии прогоном через нее
# либо разбивать на кластеры и отдельно каждый кластер прогонять через сентимент модель.
# после чего для интерпретируемости называть классы по самым популярным настроениям
# так получим более адаптированную группировку по кластерам + интерпретируемость

# K = 5
# kmeans = KMeans(n_clusters=K, random_state=42)
# cluster_ids = kmeans.fit_predict(comment_embs)

# cur_idx = 0

# ft_dataset = []
# for post_idx, pairs in enumerate(corpus):
#   post = pairs[0]
#   comments_pair = pairs[1]
#   for comment in comments_pair:
#     ft_dataset.append([post, cluster_ids[cur_idx], comment])
#     cur_idx += 1

examples = [
    {
        # "input_text": f"Пост: {post} Тип реакции: {cluster}",
        "input_text": f"Пост: {post}",
        "output_text": comment
    }
    for post, comment in corpus
]

## 2. Препроцессинг

In [ ]:
model_name = "sberbank-ai/rugpt3small_based_on_gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
# определяем токен окончания генерации
if tokenizer.eos_token is None:
    tokenizer.eos_token = "</s>"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)

# препроцесс
MAX_POST_LEN = 128
MAX_TOTAL_LEN = 192

def preprocess(ex):
    prompt = ex["input_text"] + "\nКомментарий: "
    target = ex["output_text"] + tokenizer.eos_token

    prompt_ids = tokenizer(
        prompt,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_POST_LEN,
    )["input_ids"]

    target_ids = tokenizer(
        target,
        add_special_tokens=False,
        truncation=False,
    )["input_ids"]

    input_ids = prompt_ids + target_ids
    labels = [-100] * len(prompt_ids) + target_ids

    # финальное ограничение длины
    input_ids = input_ids[:MAX_TOTAL_LEN]
    labels = labels[:MAX_TOTAL_LEN]

    # ручной паддинг
    pad_len = MAX_TOTAL_LEN - len(input_ids)
    if pad_len > 0:
        input_ids += [tokenizer.pad_token_id] * pad_len
        labels += [-100] * pad_len

    return {
        "input_ids": input_ids,
        "labels": labels,
        "attention_mask": [1] * (MAX_TOTAL_LEN - pad_len) + [0] * pad_len
    }
# добавление спецтокенов для кластеров
# special_tokens = {"additional_special_tokens": [f"<cluster_{i}>" for i in range(K)]}
# num_added = tokenizer.add_special_tokens(special_tokens)
# model.resize_token_embeddings(len(tokenizer))

In [ ]:
dataset = Dataset.from_list(examples)
dataset = dataset.map(preprocess)

# разбиваем выборку
train_test = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = train_test["train"]
val_test = train_test["test"].train_test_split(test_size=0.5, seed=42)
val_dataset, test_dataset = val_test["train"], val_test["test"]
# задаем параметры обучения модели
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    # gradient_accumulation_steps=4,
    per_device_eval_batch_size=8,
    num_train_epochs=7,
    weight_decay=0.001,
    lr_scheduler_type="cosine",
    logging_dir="./logs",
    logging_steps=10,
    push_to_hub=False,
    fp16=torch.cuda.is_available(),
    report_to=["tensorboard"],
)

# создаем хф-трейнер
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

In [ ]:
# запуск обучение
trainer.train()

# сохранение модели
save_dir = "/content/drive/MyDrive/diploma/weights_model"

trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

In [ ]:
input()

## Проверим результат на примере

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

load_dir = "/content/drive/MyDrive/diploma/weights_model"

tokenizer = AutoTokenizer.from_pretrained(load_dir)
model = AutoModelForCausalLM.from_pretrained(load_dir).eval()

In [ ]:
post_text = """В октябре световой день в Москве сократится на 2 часа 15 минут.

По данным Московского планетария, в начале месяца солнце будет светить 11 часов 34 минуты, а к концу — только 9 часов 19 минут."""
post_text = test_dataset["input_text"][225]
# cluster_id = 1  # тип реакции
prompt = f"{post_text} \nКомментарий:"

In [ ]:
test_dataset["input_text"][225]

In [ ]:
test_dataset["output_text"][225]

In [ ]:
# если загружать веса модели без обучения, то инпут должен быть на цпу, если сразу после обучения, то нужно перекинуть на cuda
inputs = tokenizer(prompt, return_tensors="pt")#.to(device)

outputs = model.generate(
    **inputs,
    max_new_tokens=210,
    top_p=0.9,
    no_repeat_ngram_size=3,
    repetition_penalty=1.2,
    temperature=0.7,
    pad_token_id=tokenizer.eos_token_id
)

generated_text = tokenizer.decode(outputs[0], skip_special_tokens=False)
print(generated_text)

In [ ]:
# load_dir_1e = "/content/drive/MyDrive/diploma/weights_model_with_1_epoch"
save_dir_4e = "/content/drive/MyDrive/diploma/weights_model_with_4_epoch"

trainer.save_model(save_dir_4e)
tokenizer.save_pretrained(save_dir_4e)

In [ ]:
sdir_1e = "/content/drive/MyDrive/diploma/weights_model_with_1_epoch"

checkpoint = "sberbank-ai/rugpt3small_based_on_gpt2"

model1 = AutoModelForCausalLM.from_pretrained(checkpoint)